# Demo — Every Component End-to-End

Walks through each capability the project provides. Run after `make pipeline` has produced all artifacts in `data/processed/` and `data/features/`.

| Step | What it shows |
|---|---|
| 1 | Raw → clean transactions |
| 2 | Time-window churn labels |
| 3 | Customer features (baseline & expanded) |
| 4 | Three churn models — load scores, compare |
| 5 | SASRec user vector + FAISS retrieval |
| 6 | LightGBM ranker rerank |
| 7 | NeuMF ranker (alternative) |
| 8 | LLM reranker (if API key) |
| 9 | Decision layer — produce a retention action |
| 10 | Final ablation: churn-only vs full pipeline |

In [ ]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

PROC = PROJECT_ROOT / 'data' / 'processed'
FEAT = PROJECT_ROOT / 'data' / 'features'
print('processed files:', [p.name for p in PROC.glob('*.parquet')])
print('feature files: ', [p.name for p in FEAT.glob('*.parquet')])

## 1. Cleaned transactions

In [ ]:
df = pd.read_parquet(PROC / 'transactions_clean.parquet')
print(f'{len(df):,} rows, {df["customer_id"].nunique():,} customers, {df["stock_code"].nunique():,} products')
df.head(3)

## 2. Churn labels per window

In [ ]:
for n in ['train','val','test']:
    t = pd.read_parquet(PROC / f'churn_labels_{n}.parquet')
    print(f'{n:>5}: {len(t):>6,} customers  |  churn rate = {t["churn"].mean()*100:.2f}%')
t.head(3)

## 3. Customer features — baseline & expanded

In [ ]:
base = pd.read_parquet(FEAT / 'baseline_features_test.parquet')
expd = pd.read_parquet(FEAT / 'expanded_features_test.parquet')
print(f'baseline: {base.shape}\nexpanded: {expd.shape}')
expd.iloc[:1].T.head(20)

## 4. Three churn models compared

In [ ]:
scores = {
    'GBM': pd.read_parquet(FEAT / 'churn_scores_gbm_test.parquet'),
    'BG-NBD': pd.read_parquet(FEAT / 'churn_scores_bgnbd_test.parquet'),
    'Cox': pd.read_parquet(FEAT / 'churn_scores_cox_test.parquet'),
}
for k, v in scores.items():
    print(f'{k:>7}: {len(v):>6,} customers,  p mean={v.iloc[:,1].mean():.3f}, p std={v.iloc[:,1].std():.3f}')

## 5. SASRec retrieval

In [ ]:
from src.models.retrieval.sasrec import SASRec
from src.models.retrieval.dataset import pad_right
from src.faiss.index import load_index, topk, l2_normalize

vocab = torch.load(FEAT / 'sasrec' / 'vocab.pt', weights_only=False)
vocab_size = len(vocab['id2item']) + 2
model = SASRec(vocab_size=vocab_size, max_len=50)
model.load_state_dict(torch.load(FEAT / 'sasrec' / 'sasrec.pt', map_location='cpu'))
model.eval()
index = load_index(FEAT / 'item_index.faiss')

# Take one real customer's history
cust_id = int(df['customer_id'].iloc[0])
hist = df[df['customer_id']==cust_id].sort_values('invoice_date')
enc = [vocab['item2id'][i] for i in hist['stock_code'].tolist() if i in vocab['item2id']]
x = torch.tensor([pad_right(enc, 50, vocab['pad_id'])], dtype=torch.long)
with torch.no_grad():
    uvec = l2_normalize(model.user_vector(x).cpu().numpy())
scores_, ids_ = topk(index, uvec, k=10)
id2item = vocab['id2item']
top10 = pd.DataFrame({
    'rank': range(10),
    'stock_code': [id2item[int(i)] if 0<=int(i)<len(id2item) else None for i in ids_[0]],
    'score': scores_[0],
})
print(f'Top-10 retrieval for customer {cust_id}:')
top10

## 6. LightGBM ranker — load retrieval candidates and feature-join

In [ ]:
cand = pd.read_parquet(FEAT / 'retrieval_candidates_test.parquet')
print(f'{len(cand):,} candidate rows across {cand["customer_id"].nunique():,} customers')
cand[cand['customer_id']==cust_id].head(5)

## 7. NeuMF ranker — quick train on positives + sampled negatives

In [ ]:
from src.models.ranking.neumf import NeuMFConfig, train_neumf, score as neumf_score
from src.data.splits import build_windows

windows = build_windows(df, horizon_days=90, n_windows=3)
lw = df[(df['invoice_date'] >= windows[0].feature_end) & (df['invoice_date'] < windows[0].label_end)]
positives = lw[['customer_id','stock_code']].drop_duplicates()
print(f'Training NeuMF on {len(positives):,} positive (customer, item) pairs ...')
neumf, user2id, item2id = train_neumf(positives, NeuMFConfig(epochs=3, batch_size=2048))
print('NeuMF trained.')

## 8. LLM reranker — if API key is set

In [ ]:
if os.environ.get('ANTHROPIC_API_KEY'):
    from src.models.reranker.llm import LLMReranker, LLMRerankerConfig
    rer = LLMReranker(LLMRerankerConfig(top_k_to_rerank=10))
    hist_d = df[df['customer_id']==cust_id].tail(20)[['stock_code','description']]
    cand_d = top10.merge(
        df[['stock_code','description','price']].drop_duplicates('stock_code'),
        on='stock_code', how='left'
    ).fillna({'description':'?','price':0.0})
    ordered = rer.rerank(hist_d, cand_d)
    print('LLM reranked top-10:')
    print(ordered)
else:
    print('No ANTHROPIC_API_KEY in env — set it to demo LLM reranker.')

## 9. Decision layer — final retention action

In [ ]:
from src.decision.retention import decide
p = float(scores['GBM'].set_index('customer_id').loc[cust_id, 'p_churn_gbm'])
action = decide(cust_id, p, top10['stock_code'].astype(str).tolist())
action

## 10. The reports — all final ablations + comparisons

In [ ]:
REPORTS = PROJECT_ROOT / 'reports'
for csv in sorted(REPORTS.glob('*.csv')):
    print(f'\n=== {csv.name} ===')
    print(pd.read_csv(csv).to_string())